# E-Commerce Recommendation System

End-to-end demo using **Ray Data → Ray Train → Ray Serve** on Anyscale.

```
┌──────────────────────────────────────────────────────────────────────┐
│  Product catalog (synthetic)                                         │
│       │                                                              │
│       ▼  Stage 1 — Ray Data                                         │
│  Preprocess images (resize, normalise)                               │
│  Clean text (descriptions + category)                               │
│       │                                                              │
│       ▼  Stage 2 — Ray Train (TorchTrainer)                         │
│  Fine-tune all-MiniLM-L6-v2 with contrastive loss                  │
│  Save model + pre-compute product embeddings                        │
│       │                                                              │
│       ▼  Stage 3 — Ray Serve                                        │
│  POST /recommend  {image_base64}                                     │
│    ImageToText  (BLIP-base)  →  caption                             │
│    ProductRecommender  (MiniLM)  →  top-5 products                  │
│       │                                                              │
│       ▼  Streamlit UI                                               │
│  Upload image → see recommendations                                 │
└──────────────────────────────────────────────────────────────────────┘
```

**Models used**
| Stage | Model | Size | Device |
|-------|-------|------|--------|
| Train | `sentence-transformers/all-MiniLM-L6-v2` | 22 M params | CPU |
| Serve | `Salesforce/blip-image-captioning-base` | 224 M params | CPU |
| Serve | fine-tuned MiniLM | 22 M params | CPU |

Everything runs on **CPU** and completes in a few minutes.

## 0 — Setup

In [ ]:
# Install dependencies (skip if already installed)
# !pip install -q -r setup/requirements.txt

In [ ]:
import json
import os
import sys
import numpy as np
from pathlib import Path
from PIL import Image

# Make sure the project root is on the path
sys.path.insert(0, str(Path().resolve()))

from utils import PRODUCTS, CATEGORIES, make_product_image

print(f"Products in catalog : {len(PRODUCTS)}")
print(f"Categories          : {CATEGORIES}")

### Peek at the synthetic product images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, cat in zip(axes, CATEGORIES):
    product = next(p for p in PRODUCTS if p["category"] == cat)
    img = make_product_image(product, seed=0)
    ax.imshow(img)
    ax.set_title(cat, fontsize=9)
    ax.axis("off")
plt.suptitle("One sample image per category", fontsize=12)
plt.tight_layout()
plt.show()

---
## Stage 1 — Ray Data: Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Stage 1 — Ray Data Preprocessing Pipeline
#
# 1. Generate synthetic product catalog (name, category, description, image)
# 2. Load into Ray Data
# 3. Preprocess images (resize → normalise → float32 bytes)
# 4. Clean text
# 5. Write Parquet to data/preprocessed/
# ─────────────────────────────────────────────────────────────────────
import time
import ray
from utils import generate_catalog, preprocess_image_batch, preprocess_text_batch

OUTPUT_DIR = os.path.abspath("data/preprocessed")
BATCH_SIZE = 8

ray.init(ignore_reinit_error=True)
ctx = ray.data.DataContext.get_current()
ctx.enable_progress_bars = True

t0 = time.time()
print("=" * 60)
print("STAGE 1 — RAY DATA: PREPROCESSING")
print("=" * 60)

# 1. Generate catalog
print("\n[1/4] Generating synthetic product catalog …")
records = generate_catalog(products=PRODUCTS, output_dir=os.path.abspath("data/raw"))
ds = ray.data.from_items(records)
print(f"  Rows: {ds.count()}")

# 2. Preprocess images
print("\n[2/4] Preprocessing images (resize + normalise) …")
ds = ds.map_batches(
    preprocess_image_batch,
    batch_size=BATCH_SIZE,
    num_cpus=1,
    batch_format="numpy",
)

# 3. Preprocess text
print("\n[3/4] Preprocessing text (clean) …")
ds = ds.map_batches(
    preprocess_text_batch,
    batch_size=BATCH_SIZE,
    num_cpus=1,
    batch_format="numpy",
)

# 4. Write Parquet
print(f"\n[4/4] Writing to '{OUTPUT_DIR}' …")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
ds.write_parquet(OUTPUT_DIR)

result_ds = ray.data.read_parquet(OUTPUT_DIR)
print(f"\nPreprocessing complete!  Rows written: {result_ds.count()}  ({time.time()-t0:.1f}s)")
print("=" * 60)

In [ ]:
# Inspect the preprocessed dataset
ds = ray.data.read_parquet("data/preprocessed")
print(f"Rows: {ds.count()}")
print(f"Schema:\n{ds.schema()}")

sample = ds.take(2)
for row in sample:
    print(f"  [{row['product_id']}] {row['name']!r}  "
          f"img_bytes={len(row['image_tensor_bytes'])}  "
          f"text={row['text_clean'][:60]}…")

---
## Stage 2 — Ray Train: Embedding Fine-Tuning

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Stage 2 — Training definitions
#
# Fine-tune sentence-transformers/all-MiniLM-L6-v2 with contrastive loss.
# Products in the same category are pulled together; cross-category apart.
# ─────────────────────────────────────────────────────────────────────
import tempfile
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from ray.train import (
    Checkpoint, CheckpointConfig, FailureConfig,
    RunConfig, ScalingConfig, get_checkpoint, get_context, report,
)
from ray.train.torch import TorchTrainer

# --- Configuration ---
_HERE = os.path.abspath(".")
PREPROCESSED_DIR = os.path.join(_HERE, "data", "preprocessed")
MODEL_OUTPUT_DIR = os.path.join(_HERE, "models", "embedding_model")
EMBEDDINGS_PATH  = os.path.join(_HERE, "models", "product_embeddings.npy")
METADATA_PATH    = os.path.join(_HERE, "models", "product_metadata.json")

BASE_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
NUM_EPOCHS    = 2
BATCH_SIZE_TRAIN = 8
LEARNING_RATE = 2e-5
SEED          = 42

TRAIN_RESULT_DIR = (
    "/mnt/cluster_storage/ecomm_ray_train_results"
    if os.path.isdir("/mnt/cluster_storage")
    else os.path.join(_HERE, "models", "ray_train_results")
)


# --- Contrastive pair dataset ---
class ContrastivePairDataset(Dataset):
    """
    (anchor, positive, 1.0) for same-category pairs;
    (anchor, negative, -1.0) for cross-category pairs.
    """
    def __init__(self, records, neg_ratio=0.5, seed=SEED):
        rng = np.random.default_rng(seed)
        cat_to_idx = {}
        for i, r in enumerate(records):
            cat_to_idx.setdefault(r["category"], []).append(i)

        pairs = []
        for indices in cat_to_idx.values():
            for i, a in enumerate(indices):
                for b in indices[i + 1:]:
                    pairs.append((records[a]["text_clean"], records[b]["text_clean"], 1.0))

        cats = list(cat_to_idx.keys())
        n_neg = max(1, int(len(pairs) * neg_ratio))
        for _ in range(n_neg):
            cat_a, cat_b = rng.choice(cats, size=2, replace=False)
            ia = rng.choice(cat_to_idx[cat_a])
            ib = rng.choice(cat_to_idx[cat_b])
            pairs.append((records[ia]["text_clean"], records[ib]["text_clean"], -1.0))

        rng.shuffle(pairs)
        self.pairs = pairs

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        a, b, label = self.pairs[idx]
        return a, b, torch.tensor(label, dtype=torch.float32)


# --- Forward pass with gradient tracking ---
def _forward_embeddings(model, texts, device):
    features = model.tokenize(texts)
    features = {k: v.to(device) for k, v in features.items()}
    return model(features)["sentence_embedding"]


# --- Per-worker training loop (runs inside each Ray Train worker) ---
def train_loop_per_worker(config):
    from sentence_transformers import SentenceTransformer

    rank   = get_context().get_world_rank()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    records = config["records"]

    pair_dataset = ContrastivePairDataset(records, seed=config["seed"])

    def collate(batch):
        texts_a, texts_b, labels = zip(*batch)
        return list(texts_a), list(texts_b), torch.stack(labels)

    loader = DataLoader(
        pair_dataset, batch_size=config["batch_size"],
        shuffle=True, collate_fn=collate,
    )

    model = SentenceTransformer(config["base_model"], device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=0.01)

    start_epoch = 0
    ckpt = get_checkpoint()
    if ckpt:
        with ckpt.as_directory() as d:
            meta = torch.load(os.path.join(d, "meta.pt"), map_location="cpu")
            start_epoch = meta.get("epoch", 0) + 1
            model = SentenceTransformer(d, device=device)

    if rank == 0:
        print(f"Fine-tuning {config['base_model']} on {device}  "
              f"({config['epochs']} epochs, {len(pair_dataset)} pairs)")

    for epoch in range(start_epoch, config["epochs"]):
        model.train()
        total_loss, n_batches = 0.0, 0
        for texts_a, texts_b, labels in loader:
            labels = labels.to(device)
            emb_a  = _forward_embeddings(model, texts_a, device)
            emb_b  = _forward_embeddings(model, texts_b, device)
            loss   = F.mse_loss(F.cosine_similarity(emb_a, emb_b), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(1, n_batches)
        if rank == 0:
            print(f"  Epoch {epoch+1:2d}/{config['epochs']}  loss={avg_loss:.4f}")

        with tempfile.TemporaryDirectory() as tmpdir:
            if rank == 0:
                model.save(tmpdir)
                torch.save({"epoch": epoch}, os.path.join(tmpdir, "meta.pt"))
                ckpt_out = Checkpoint.from_directory(tmpdir)
            else:
                ckpt_out = None
            report({"epoch": epoch, "train_loss": avg_loss}, checkpoint=ckpt_out)


print("Training definitions ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Stage 2 — Launch TorchTrainer and export model + embeddings
# ─────────────────────────────────────────────────────────────────────
import glob as _glob
import pandas as _pd
from sentence_transformers import SentenceTransformer

ray.init(ignore_reinit_error=True)
t0 = time.time()

print("=" * 60)
print("STAGE 2 — RAY TRAIN: EMBEDDING FINE-TUNING")
print("=" * 60)

# Load preprocessed records (34 rows — small enough to pass via config)
parquet_files = sorted(_glob.glob(f"{PREPROCESSED_DIR}/*.parquet"))
df = _pd.concat([_pd.read_parquet(f) for f in parquet_files], ignore_index=True)
records = df[["product_id", "name", "category", "text_clean"]].to_dict(orient="records")
print(f"\nLoaded {len(records)} records for training")

train_config = {
    "base_model": BASE_MODEL,
    "epochs":     NUM_EPOCHS,
    "batch_size": BATCH_SIZE_TRAIN,
    "lr":         LEARNING_RATE,
    "seed":       SEED,
    "records":    records,
}

Path(TRAIN_RESULT_DIR).mkdir(parents=True, exist_ok=True)

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_config,
    scaling_config=ScalingConfig(
        num_workers=1,
        use_gpu=torch.cuda.is_available(),
    ),
    run_config=RunConfig(
        name="ecomm_embedding_finetune",
        storage_path=os.path.abspath(TRAIN_RESULT_DIR),
        checkpoint_config=CheckpointConfig(
            num_to_keep=2,
            checkpoint_score_attribute="train_loss",
            checkpoint_score_order="min",
        ),
        failure_config=FailureConfig(max_failures=1),
    ),
)

result = trainer.fit()
print(f"\nTraining complete! ({time.time()-t0:.1f}s)")

# --- Export model and compute product embeddings ---
print("\nExporting model and computing product embeddings …")
Path(MODEL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

checkpoint = result.best_checkpoints[0][0]
with checkpoint.as_directory() as ckpt_dir:
    model = SentenceTransformer(ckpt_dir)
    model.save(MODEL_OUTPUT_DIR)
print(f"  Model saved : {MODEL_OUTPUT_DIR}")

all_records = (
    ray.data.read_parquet(PREPROCESSED_DIR)
    .select_columns(["product_id", "name", "category", "text_clean"])
    .take_all()
)
texts      = [r["text_clean"] for r in all_records]
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

np.save(EMBEDDINGS_PATH, embeddings)
metadata = [
    {"product_id": r["product_id"], "name": r["name"], "category": r["category"]}
    for r in all_records
]
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"  Embeddings  : {EMBEDDINGS_PATH}  shape={embeddings.shape}")
print(f"  Metadata    : {METADATA_PATH}  ({len(metadata)} products)")
print("=" * 60)

In [ ]:
# Inspect the computed embeddings
embeddings = np.load("models/product_embeddings.npy")
with open("models/product_metadata.json") as f:
    metadata = json.load(f)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Products indexed : {len(metadata)}")

# Cosine similarity heatmap between first 10 products
n = min(10, len(embeddings))
sub = embeddings[:n]
norms = np.linalg.norm(sub, axis=1, keepdims=True)
sim = (sub / norms) @ (sub / norms).T

labels = [m["name"][:20] for m in metadata[:n]]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim, vmin=-1, vmax=1, cmap="RdYlGn")
ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_title("Product embedding similarity (top 10)")
plt.tight_layout()
plt.show()

---
## Stage 3 — Ray Serve: Online Recommendation API

In [ ]:
import ray
from ray import serve

ray.init(ignore_reinit_error=True)

# Deploy locally
from serve_app import app
serve.run(app, host="0.0.0.0", port=8000)
print("Service running at http://localhost:8000")

In [ ]:
# --- Test the endpoint ---
import base64, io, requests
from utils import make_product_image, PRODUCTS

# Build a test image (Wireless Headphones)
product = PRODUCTS[0]
img_arr = make_product_image(product, seed=0)
img_pil = Image.fromarray(img_arr)

buf = io.BytesIO()
img_pil.save(buf, format="JPEG")
encoded = base64.b64encode(buf.getvalue()).decode()

resp = requests.post("http://localhost:8000/recommend", json={"image_base64": encoded})
result = resp.json()

print(f"Caption : {result['caption']!r}")
print(f"\nTop-{len(result['recommendations'])} recommendations:")
for r in result["recommendations"]:
    print(f"  {r['rank']}. [{r['category']:18s}] {r['name']:35s}  sim={r['similarity']:.3f}")

In [ ]:
# Show the query image alongside recommendations
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

axes[0].imshow(img_arr)
axes[0].set_title("Query image", fontsize=9, color="navy", fontweight="bold")
axes[0].axis("off")

with open("models/product_metadata.json") as f:
    meta = {m["product_id"]: m for m in json.load(f)}

all_prods = {p["name"]: p for p in PRODUCTS}

for ax, rec in zip(axes[1:], result["recommendations"]):
    prod_name = rec["name"]
    prod = all_prods.get(prod_name, {"name": prod_name, "category": rec["category"]})
    rec_img = make_product_image({**prod, "category": rec["category"]}, seed=42)
    ax.imshow(rec_img)
    ax.set_title(
        f"{rec['rank']}. {rec['name'][:18]}\n{rec['similarity']:.2f}",
        fontsize=8
    )
    ax.axis("off")

plt.suptitle(f'Caption: "{result["caption"]}"', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Teardown (optional — keeps the service running if you want to use the Streamlit app)
# serve.shutdown()

---
## Streamlit UI

With the service running, open a terminal and run:

```bash
streamlit run streamlit_app.py
```

Then navigate to **http://localhost:8501** in your browser.

---

## Deploy to Anyscale

```bash
# Stage 1 — preprocess
anyscale job submit -f setup/job_preprocess.yaml

# Stage 2 — train
anyscale job submit -f setup/job_train.yaml

# Stage 3 — serve
anyscale service deploy -f setup/service.yaml
```